# 04 — 模型结果可视化

## 目标

可视化启发式调度优化模型的求解结果，回答以下问题：

1. **求解成功了吗？** — 对比 3 个网格尺度（1km / 500m / 200m）的求解状态和效果
2. **调度车怎么走？** — 在地图上画出每辆车的路线，用箭头标出行驶方向
3. **在哪装车、在哪卸车？** — 在地图上标注 pickup/dropoff 点位
4. **短缺都满足了吗？** — 热力图展示每个网格的满足率
5. **车辆工作量均衡吗？** — 对比 20 辆车的工时
6. **目标函数由哪些部分构成？** — 分解 makespan / 总工时 / 未满足量

## 数据来源

| 文件 | 内容 |
|---|---|
| `outputs/model_results/summary_*.csv` | 求解状态和目标值 |
| `outputs/model_results/routes_*.csv` | 车辆行驶路线边表 |
| `outputs/model_results/vehicle_actions_*.csv` | 装卸车动作表 |
| `outputs/model_results/shortage_service_*.csv` | 短缺满足情况 |
| `outputs/model_results/node_copies_*.csv` | 副本节点→真实网格映射 |
| `outputs/model_results/vehicle_time_*.csv` | 每辆车总工时 |
| `outputs/model_results/objective_parts_*.csv` | 目标函数分解 |
| `data/processed/grid_metadata_*.csv` | 网格经纬度 |
| `data/processed/dispatch_vehicle_params_1km.csv` | 调度中心 (DEPOT) 坐标 |

## 关键概念

- **副本节点 (copy node)**：模型为允许同一网格被多次访问，将大需求网格拆分成 `__v01`, `__v02` 等副本。画地图时必须通过 `node_copies` 映射回 `original_grid_id`
- **DEPOT**：所有调度车的起点和终点，统一位于福田区中心 (114.0627, 22.5394)
- **3 个网格尺度**：1km (粗) / 500m (中) / 200m (细)，`summary` 文件可对比不同尺度的求解效果

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import FancyArrowPatch, Patch

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 调度中心坐标（来自 dispatch_vehicle_params_1km.csv）
DEPOT_LNG = 114.06266855872391
DEPOT_LAT = 22.539407114624506

print('库加载完成')

In [ ]:
# 全局配置：当前分析场景和尺度
SCENARIO = '20210512_pm_peak'  # 模型只跑了这个场景
GRID = '200m'                   # 可选 1000m, 500m, 200m

# 读取所有模型输出
MODEL = '../outputs/model_results'
summary = pd.read_csv(f'{MODEL}/summary_{GRID}_{SCENARIO}.csv')
routes = pd.read_csv(f'{MODEL}/routes_{GRID}_{SCENARIO}.csv')
actions = pd.read_csv(f'{MODEL}/vehicle_actions_{GRID}_{SCENARIO}.csv')
shortage = pd.read_csv(f'{MODEL}/shortage_service_{GRID}_{SCENARIO}.csv')
node_copies = pd.read_csv(f'{MODEL}/node_copies_{GRID}_{SCENARIO}.csv')
veh_time = pd.read_csv(f'{MODEL}/vehicle_time_{GRID}_{SCENARIO}.csv')
obj_parts = pd.read_csv(f'{MODEL}/objective_parts_{GRID}_{SCENARIO}.csv')

# 网格经纬度（注意 1km 文件名不同）
if GRID == '1000m':
    meta = pd.read_csv('../data/processed/grid_metadata_1km.csv')
else:
    meta = pd.read_csv(f'../data/processed/grid_metadata_{GRID}.csv')

print(f'场景: {SCENARIO}, 网格尺度: {GRID}')
print(f'求解状态: {summary["status"][0]}')
print(f'目标值: {summary["objective"][0]:,.1f}')
print(f'车辆数: {routes["vehicle_id"].nunique()}')
print(f'路线边数: {len(routes):,}')
print(f'装卸动作数: {len(actions):,}')
print(f'服务节点数: {len(shortage):,}')

---

## 1. 求解摘要 — 3 个尺度对比

### 这是什么

对比 1km / 500m / 200m 三个网格尺度下模型的求解状态、目标值和运行时间。网格越细 → 问题规模越大 → 求解越难。

### 关键指标

| 字段 | 含义 |
|---|---|
| `status` | 求解状态：`heuristic_feasible`=找到可行解，`heuristic_service_level_not_met`=有解但未完全满足服务水平 |
| `objective` | 目标函数值 = alpha×makespan + gamma×总工时 + beta×未满足量 |
| `runtime_seconds` | 求解耗时 |
| `nodes_explored` | 模拟退火迭代次数 |

In [ ]:
# 读取 3 个尺度的 summary
summaries = {}
for g in ['1000m', '500m', '200m']:
    s = pd.read_csv(f'{MODEL}/summary_{g}_{SCENARIO}.csv')
    summaries[g] = s

# 对比表
compare = pd.DataFrame({
    'Scale': ['1km (coarse)', '500m (medium)', '200m (fine)'],
    'Status': [summaries[g]['status'][0] for g in ['1000m','500m','200m']],
    'Objective': [summaries[g]['objective'][0] for g in ['1000m','500m','200m']],
    'Runtime (s)': [summaries[g]['runtime_seconds'][0] for g in ['1000m','500m','200m']],
    'Iterations': [summaries[g]['nodes_explored'][0] for g in ['1000m','500m','200m']],
})
compare

In [ ]:
# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 目标值
axes[0].bar(compare['Scale'], compare['Objective'], color=['#4575B4','#F0A030','#D73027'])
axes[0].set_title('Objective Value\n(lower is better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Objective')

# 运行时间
axes[1].bar(compare['Scale'], compare['Runtime (s)'], color=['#4575B4','#F0A030','#D73027'])
axes[1].set_title('Runtime (seconds)\n(longer for finer grids)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Seconds')

# 状态（文字标注）
status_colors = ['#55A868' if 'feasible' in s and 'not_met' not in s else '#E67E22' for s in compare['Status']]
axes[2].bar(compare['Scale'], [1,1,1], color=status_colors)
for i, (s, c) in enumerate(zip(compare['Status'], status_colors)):
    label = 'FEASIBLE' if c == '#55A868' else 'PARTIALLY MET'
    axes[2].text(i, 0.5, label, ha='center', va='center', fontweight='bold', fontsize=11, color='white')
axes[2].set_title('Solution Status', fontsize=12, fontweight='bold')
axes[2].set_ylim(0, 1.2); axes[2].set_yticks([])

fig.suptitle(f'Model Comparison Across 3 Grid Scales — {SCENARIO}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'../outputs/figures/04_summary_compare.png', dpi=150, bbox_inches='tight')
plt.show()

> **观察**：1km 和 500m 获得了可行解（heuristic_feasible），200m 虽然找到了路线但未完全满足服务水平约束。网格越细，问题越复杂，求解时间从 29s 增长到 691s。

---

## 2. 车辆行驶路线地图

### 这是什么

在地图上画出**每辆调度车的行驶路线**，用箭头标出方向（DEPOT → 网格1 → 网格2 → ... → DEPOT）。

### 怎么看

| 元素 | 含义 |
|---|---|
| **彩色箭头** | 不同车辆的路线，箭头指向行驶方向 |
| **绿色方块** | DEPOT（调度中心），所有路线的起点和终点 |
| **灰点** | 所有涉及的服务网格位置 |

In [ ]:
# 将 node_copies 的 grid_id (可能带 __v01) 映射到 original_grid_id
copy_map = dict(zip(node_copies['grid_id'], node_copies['original_grid_id']))

# 对 from/to 做映射（DEPOT 保持 DEPOT）
routes['from_original'] = routes['from_grid_id'].map(copy_map).fillna(routes['from_grid_id'])
routes['to_original'] = routes['to_grid_id'].map(copy_map).fillna(routes['to_grid_id'])

# 查经纬度
grid_lookup = meta.set_index('grid_id')[['center_lng', 'center_lat']]

def get_coords(gid):
    if gid == 'DEPOT':
        return (DEPOT_LNG, DEPOT_LAT)
    if gid in grid_lookup.index:
        return (grid_lookup.at[gid, 'center_lng'], grid_lookup.at[gid, 'center_lat'])
    return (None, None)

routes['from_lng'], routes['from_lat'] = zip(*routes['from_original'].map(get_coords))
routes['to_lng'], routes['to_lat'] = zip(*routes['to_original'].map(get_coords))

vehicles = sorted(routes['vehicle_id'].unique())
print(f'共 {len(vehicles)} 辆车: {vehicles[:5]}...{vehicles[-2:]}')
print(f'总路线边: {len(routes)}')
print(f'总行驶距离: {routes["distance_km"].sum():.1f} km')

In [ ]:
# 画全 20 辆车的路线图（不同颜色区分）
fig, ax = plt.subplots(figsize=(18, 14))

# 背景：所有被访问的网格（灰点）
all_visited = set(routes['from_original'].unique()) | set(routes['to_original'].unique())
all_visited.discard('DEPOT')
visited_coords = meta[meta['grid_id'].isin(all_visited)]
ax.scatter(visited_coords['center_lng'], visited_coords['center_lat'],
           s=3, c='#D0D0D0', alpha=0.5, edgecolors='none', zorder=1)

# 每辆车用不同颜色
cmap = plt.cm.tab20
for i, vid in enumerate(vehicles):
    v_routes = routes[routes['vehicle_id'] == vid]
    color = cmap(i % 20)
    
    for _, row in v_routes.iterrows():
        if pd.isna(row['from_lng']) or pd.isna(row['to_lng']):
            continue
        lw = max(0.5, row['distance_km'] * 2)
        arrow = FancyArrowPatch(
            (row['from_lng'], row['from_lat']),
            (row['to_lng'], row['to_lat']),
            arrowstyle='-|>', mutation_scale=8 + lw * 3,
            color=color, linewidth=lw, alpha=0.7, zorder=2
        )
        ax.add_patch(arrow)

# DEPOT 绿色方块
ax.scatter([DEPOT_LNG], [DEPOT_LAT], s=200, c='#1A9850', marker='s',
           edgecolors='#006837', linewidth=2, zorder=3, label='DEPOT (Start/End)')

ax.set_title(f'Vehicle Routes — {GRID} {SCENARIO}\n{len(vehicles)} vehicles, {len(routes)} route segments',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(f'../outputs/figures/04_routes_map_{GRID}_{SCENARIO}.png', dpi=200, bbox_inches='tight')
plt.show()

---

## 3. 装卸车动作地图

### 这是什么

在地图上标注每辆车在哪里**装车**（从富余网格取车，蓝色+号）和**卸车**（送到短缺网格，红色-号）。

### 怎么看

| 符号 | 含义 |
|---|---|
| 🔵 蓝色圆圈 | Pickup（装车点），圆圈大小 ∝ 装车数量 |
| 🔴 红色圆圈 | Dropoff（卸车点），圆圈大小 ∝ 卸车数量 |

In [ ]:
# 将 actions 的 grid_id 映射到原始网格
actions['original_grid_id'] = actions['grid_id'].map(copy_map).fillna(actions['grid_id'])

# 分别聚合 pickup 和 dropoff 到真实网格
pickup_agg = actions[actions['pickup_bikes'] > 0].groupby('original_grid_id')['pickup_bikes'].sum()
dropoff_agg = actions[actions['dropoff_bikes'] > 0].groupby('original_grid_id')['dropoff_bikes'].sum()

# 合并经纬度
pickup_df = pickup_agg.reset_index().merge(meta[['grid_id','center_lng','center_lat']],
                                            left_on='original_grid_id', right_on='grid_id', how='left')
dropoff_df = dropoff_agg.reset_index().merge(meta[['grid_id','center_lng','center_lat']],
                                             left_on='original_grid_id', right_on='grid_id', how='left')

print(f'Pickup 网格数: {len(pickup_df)}, 总装车: {pickup_df["pickup_bikes"].sum():.0f}')
print(f'Dropoff 网格数: {len(dropoff_df)}, 总卸车: {dropoff_df["dropoff_bikes"].sum():.0f}')
print(f'Pickup = Dropoff? {pickup_df["pickup_bikes"].sum():.0f} vs {dropoff_df["dropoff_bikes"].sum():.0f}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 11))

# 全部网格灰背景
ax.scatter(meta['center_lng'], meta['center_lat'], s=1, c='#E8E8E8', alpha=0.5, edgecolors='none', zorder=1)

# Pickup（蓝圈）
ax.scatter(pickup_df['center_lng'], pickup_df['center_lat'],
           s=pickup_df['pickup_bikes']*15, c='#2E86C1', alpha=0.7, edgecolors='#1B4F72', linewidth=0.5,
           zorder=2, label=f'Pickup ({pickup_df["pickup_bikes"].sum():.0f} bikes)')

# Dropoff（红圈）
ax.scatter(dropoff_df['center_lng'], dropoff_df['center_lat'],
           s=dropoff_df['dropoff_bikes']*15, c='#E74C3C', alpha=0.7, edgecolors='#922B21', linewidth=0.5,
           zorder=3, label=f'Dropoff ({dropoff_df["dropoff_bikes"].sum():.0f} bikes)')

# DEPOT
ax.scatter([DEPOT_LNG], [DEPOT_LAT], s=150, c='#1A9850', marker='s',
           edgecolors='#006837', linewidth=2, zorder=4, label='DEPOT')

ax.set_title(f'Pickup & Dropoff Actions — {GRID} {SCENARIO}', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude'); ax.set_aspect('equal')
ax.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig(f'../outputs/figures/04_actions_map_{GRID}_{SCENARIO}.png', dpi=200, bbox_inches='tight')
plt.show()

---

## 4. 短缺满足热力图

### 这是什么

在真实网格级别展示每个短缺点被满足的程度。聚合副本节点后，计算每个网格的 `served / shortage`。

### 怎么看

- **深绿** = 完全满足（满足率 = 100%）
- **深红** = 完全未满足（满足率 = 0%）
- **中间色** = 部分满足
- 点的大小 ∝ 短缺量（越短缺越大）

In [ ]:
# 映射到真实网格
shortage['original_grid_id'] = shortage['grid_id'].map(copy_map).fillna(shortage['grid_id'])

# 按真实网格聚合
real_short = shortage.groupby('original_grid_id').agg(
    shortage=('shortage_bikes', 'sum'),
    unmet=('unmet_bikes', 'sum'),
    served=('served_bikes', 'sum'),
    surplus=('surplus_bikes', 'sum')
).reset_index()

# 满足率
real_short['satisfaction'] = np.where(
    real_short['shortage'] > 0,
    real_short['served'] / real_short['shortage'],
    np.nan
)

# 关联经纬度
real_short = real_short.merge(meta[['grid_id','center_lng','center_lat']],
                              left_on='original_grid_id', right_on='grid_id', how='left')

# 只看有短缺的网格
real_short_only = real_short[real_short['shortage'] > 0].copy()

print(f'短缺网格数: {len(real_short_only)}')
print(f'总短缺: {real_short_only["shortage"].sum():.0f}')
print(f'总已满足: {real_short_only["served"].sum():.0f}')
print(f'总未满足: {real_short_only["unmet"].sum():.0f}')
print(f'整体满足率: {real_short_only["served"].sum()/real_short_only["shortage"].sum()*100:.1f}%')
print(f'完全满足的网格: {(real_short_only["satisfaction"]>=0.99).sum()}')
print(f'完全未满足的网格: {(real_short_only["satisfaction"]<=0.01).sum()}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 9))

# 左：满足率热力图
ax = axes[0]
ax.scatter(meta['center_lng'], meta['center_lat'], s=1, c='#E8E8E8', alpha=0.4, edgecolors='none')
sc = ax.scatter(real_short_only['center_lng'], real_short_only['center_lat'],
                c=real_short_only['satisfaction'], cmap='RdYlGn', vmin=0, vmax=1,
                s=real_short_only['shortage']*4 + 5, alpha=0.75, edgecolors='none')
cbar = plt.colorbar(sc, ax=ax, shrink=0.7)
cbar.set_label('Satisfaction Rate (served/shortage)', fontsize=10)
ax.set_title(f'Satisfaction Rate per Grid\nGreen=100% met, Red=0% met', fontsize=12, fontweight='bold')
ax.set_aspect('equal')

# 右：满足率分布直方图
ax = axes[1]
ax.hist(real_short_only['satisfaction'].dropna(), bins=20, color='#4575B4', edgecolor='white')
ax.axvline(real_short_only['satisfaction'].mean(), color='#D73027', linestyle='--', linewidth=2,
           label=f'Mean: {real_short_only["satisfaction"].mean():.1%}')
ax.set_xlabel('Satisfaction Rate', fontsize=11)
ax.set_ylabel('Number of Grids', fontsize=11)
ax.set_title('Satisfaction Rate Distribution', fontsize=12, fontweight='bold')
ax.legend()

fig.suptitle(f'Shortage Satisfaction Analysis — {GRID} {SCENARIO}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'../outputs/figures/04_satisfaction_{GRID}_{SCENARIO}.png', dpi=200, bbox_inches='tight')
plt.show()

---

## 5. 车辆工时对比

### 这是什么

对比 20 辆调度车的总服务时间，检查工作量是否均衡。差异过大的话说明路线分配不够公平。

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

veh_time_sorted = veh_time.sort_values('route_time_min')
colors = ['#4575B4' if t < veh_time['route_time_min'].mean() else '#D73027' for t in veh_time_sorted['route_time_min']]
ax.barh(range(len(veh_time_sorted)), veh_time_sorted['route_time_min'], color=colors, height=0.7)
ax.set_yticks(range(len(veh_time_sorted)))
ax.set_yticklabels(veh_time_sorted['vehicle_id'], fontsize=9, family='monospace')
ax.set_xlabel('Route Time (minutes)', fontsize=11)
ax.set_title(f'Vehicle Workload Comparison — {GRID} {SCENARIO}', fontsize=13, fontweight='bold')

# 标注统计量
mean_time = veh_time['route_time_min'].mean()
ax.axvline(mean_time, color='black', linestyle='--', linewidth=1.5, label=f'Mean: {mean_time:.1f} min')
ax.axvline(veh_time['route_time_min'].max(), color='#D73027', linestyle=':', linewidth=1, label=f'Max: {veh_time["route_time_min"].max():.1f} min')
ax.axvline(veh_time['route_time_min'].min(), color='#4575B4', linestyle=':', linewidth=1, label=f'Min: {veh_time["route_time_min"].min():.1f} min')
ax.legend(loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig(f'../outputs/figures/04_vehicle_time_{GRID}_{SCENARIO}.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 6. 目标函数分解

### 这是什么

模型的目标是最小化加权组合：`alpha × makespan + gamma × total_time + beta × unmet_bikes`。这个图展示各组成部分的实际值。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

parts = obj_parts.iloc[0]
categories = ['Makespan\n(min)', 'Total Vehicle\nTime (min)', 'Unmet\nBikes', 'Served\nBikes']
values = [parts['makespan_min'], parts['total_vehicle_time_min'], parts['total_unmet_bikes'], parts['total_served_bikes']]
bar_colors = ['#D73027', '#F0A030', '#C44E52', '#55A868']

bars = ax.bar(categories, values, color=bar_colors, edgecolor='white', width=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(values)*0.02,
            f'{val:,.1f}', ha='center', fontsize=11, fontweight='bold')

ax.set_title(f'Objective Function Components — {GRID} {SCENARIO}', fontsize=13, fontweight='bold')
ax.set_ylabel('Value')
plt.tight_layout()
plt.savefig(f'../outputs/figures/04_objective_parts_{GRID}_{SCENARIO}.png', dpi=150, bbox_inches='tight')
plt.show()

---

## 关键发现与结论

1. **尺度至关重要**：1km 和 500m 可以找到可行解，200m 由于规模太大（1,914 个网格）未能满足服务水平约束。实际应用中 500m 可能是精度与求解质量的平衡点。

2. **车辆路线覆盖广泛**：20 辆车的路线覆盖了福田区大部分失衡区域。总行驶距离和耗时在合理范围内。

3. **装卸操作匹配需求**：Pickup 和 Dropoff 总量基本相等（车辆守恒），空间分布对应短缺和富余区域。

4. **满足率不均衡**：部分网格被完全满足，部分完全未满足。这是启发式方法的局限——优先满足大短缺网格，小短缺网格可能被忽略。

5. **车辆工作量基本均衡**：工时在 120~171 分钟之间，差异约 50 分钟。进一步调优可以平衡得更好。

6. **200m 尺度未满足 592 辆**：总短缺 1,366 辆中 774 辆被满足，还有 592 辆未满足。需要更多车辆或更优的调度策略。

## 下一步

→ `05_multi_scale_comparison.ipynb` — 多尺度对比分析，或在 `docs/` 中撰写最终报告